## Random walk

A particle moves step by step at random along a straight line. It starts at position 0 and at each step moves to an adjacent position: a displacement of length 1 to the right or to the left, each with probability 0.5 (the particle cannot stay still from one step to the next).

For example, from position 0 it can go to position 1 or to position -1 with the same probability. From position 1 it can go to 0 or to 2; from -1 it can go to 0 or to -2; and so on for every position, for a walk of n steps.

At the end of the walk the particle is at position k. We want the probability that a walk of n (n ≥ 1) steps ends at position k. For more details, watch the video:
[https://www.youtube.com/watch?v=vz1wWCFpzl0&feature=youtu.be&hd=1]

Assignment 01 of the Systems Performance Modelling course (PUCPR, 2021). Team: Eduardo Eiji Goto, Gustavo Hammerschmidt, João Vitor Andrioli. The written report is in `report.pdf`.

### Iterative simulation

The function random_walk (next cell) runs nsim iterative simulations of a random walk of n_steps steps and computes the probability that the walk ends at pos. Helper functions:
* same_parity: the final position must have the same parity as the number of steps. If n_steps and pos have different parity the result must be zero and the probability formula does not apply (see the video for details). Returns True when both are even or both are odd.<br>
* combination: n choose x.<br>
* simulate_position: simulates one random walk with the given number of steps and returns the final position.

Run the simulation for several numbers of steps and final positions, varying the number of simulations between 1,000, 10,000 and 100,000. The more simulations, the better the precision.

In [70]:
# Random walk - iterative version

import time
from math import factorial
from random import randrange

def combination(n, x):
    return factorial(n)/(factorial(x)*(factorial(n - x)))

def same_parity(n, k):
    return (n % 2 == 0 and k % 2 == 0) or (n % 2 != 0 and k % 2 != 0)

def simulate_position(steps):
    position = 0
    for i in range(steps):
        side = randrange(0,2)
        if side == 0:
            # went left
            position -= 1
        else:
            # went right
            position += 1
    return position

def random_walk(n_steps, pos, nsim):
    # simulated probability
    hits = 0
    for i in range(nsim):
        if simulate_position(n_steps) == pos:
            hits += 1

    p_sim = hits/nsim

    # theoretical probability
    if same_parity(n_steps, pos):
        p_theory = combination(n_steps, (n_steps+pos)/2) * (2**(-n_steps))
    else:
        p_theory = 0

    return (p_theory, p_sim)

n_steps = 100 #int(input("Number of steps: "))
pos = 2       #int(input("Final position of the walk: "))
nsim = 1000   #int(input("Number of simulations: "))

tstart = time.perf_counter()
theoretical, simulated = random_walk(n_steps, pos, nsim)
tend = time.perf_counter()

print('Theoretical result: {0:.5f}'.format(theoretical))
print('Simulated result: {0:.5f}'.format(simulated))
print("Execution time: {:.4f}".format(tend-tstart))

Resultado teorico: 0.07803
Resultado simulado: 0.06900
Tempo de execução: 0.1336


### Vectorised simulation

The function random_walk_vectorised (next cell) runs nsim vectorised simulations of random walks of n_steps steps and computes the probability that the walk ends at pos.

Run it for several numbers of steps and final positions, varying the number of simulations between 1,000, 10,000 and 100,000, and compare with the iterative simulation. The theoretical results must be identical; for nsim = 100000 the simulated results must be very close to the iterative ones (difference below 0.01).

In [98]:
import numpy as np
from math import factorial

def combination(n, x):
    return factorial(n)/(factorial(x)*(factorial(n - x)))

def same_parity(n, k):
    return (n % 2 == 0 and k % 2 == 0) or (n % 2 != 0 and k % 2 != 0)

def random_walk_vectorised(n_steps, pos, nsim):
    # vectorised simulation

    # Step 1
    # draw a matrix with nsim rows and n_steps columns;
    # each row is one walk of n_steps steps, each step drawn as 0 (left) or 1 (right)
    matrix = np.array([ np.random.randint(0, 2, n_steps) for _ in range(nsim)])

    # Step 2
    # final position of each walk = (steps to the right) - (steps to the left)
    row_sum = [ s-(n_steps-s) for s in [ sum(_) for _ in matrix] ]

    # Step 3
    # simulated probability = walks that ended at pos / number of simulations
    p_sim = sum([1 for _ in row_sum if _==pos]) / nsim

    # theoretical probability
    if same_parity(n_steps, pos):
        p_theory = combination(n_steps, (n_steps + pos) / 2) * (2 ** (-n_steps))
    else:
        p_theory = 0

    return p_sim, p_theory

steps = 100 #int(input("Number of steps: "))
pos = 2     #int(input("Final position of the walk: "))
nsim = 1000 #int(input("Number of simulations: "))

tstart = time.perf_counter()
p_sim, p_theory = random_walk_vectorised(steps, pos, nsim)
tend = time.perf_counter()

print('Simulated probability:  {:.4f}'.format(p_sim))
print('Theoretical probability: {:.4f}'.format(p_theory))
print("Execution time: {:.4f}".format(tend-tstart))

Probabilidade simulada:  0.0580
Probabilidade teorica: 0.0780
Tempo de execução: 0.0323


In [109]:
# Comparison: theoretical vs iterative vs vectorised, for every final position of a 13-step walk

_steps = 13
_nsim = 100000
_pos = 0

import time

probabilities, times, temp, prob = [], [], [0, 0], [0, 0, 0] # prob[theory, simulated, vectorised]
for _pos in range(_steps+1):

    tstart = time.perf_counter()
    _p_theory, _p_sim = random_walk(_steps, _pos, _nsim)
    tend = time.perf_counter()

    prob[0], prob[1], temp[0] = _p_theory, _p_sim, (tend - tstart)

    tstart = time.perf_counter()
    _p_sim, _ = random_walk_vectorised(_steps, _pos, _nsim)
    tend = time.perf_counter()

    prob[2], temp[1] = _p_sim, (tend - tstart)

    times.append(temp)
    probabilities.append(prob)

    print("-"*40,"\npos:",_pos)
    print('-'*40)
    print("P[theory]     = {:.5f}".format(prob[0]))
    print("P[iterative]  = {:.5f}".format(prob[1]))
    print("P[vectorised] = {:.5f}".format(prob[2]))

    print('-'*40)
    print("Time (iterative)  = {:.5f}".format(temp[0]))
    print("Time (vectorised) = {:.5f}".format(temp[1]))
    print('-'*40, '\n')

---------------------------------------- 
pos: 0
----------------------------------------
P[teórica]     = 0.00000
P[simulada]    = 0.00000
P[simVetorial] = 0.00000
----------------------------------------
Tempo(PSimulada)    = 1.80488
Tempo(PsimVetorial) = 1.70892
---------------------------------------- 

---------------------------------------- 
pos: 1
----------------------------------------
P[teórica]     = 0.20947
P[simulada]    = 0.20987
P[simVetorial] = 0.20798
----------------------------------------
Tempo(PSimulada)    = 1.80493
Tempo(PsimVetorial) = 1.82737
---------------------------------------- 

---------------------------------------- 
pos: 2
----------------------------------------
P[teórica]     = 0.00000
P[simulada]    = 0.00000
P[simVetorial] = 0.00000
----------------------------------------
Tempo(PSimulada)    = 1.81616
Tempo(PsimVetorial) = 1.62430
---------------------------------------- 

---------------------------------------- 
pos: 3
------------------------